**First run test plan**

0. Add/patch apply_variant in ecoli/variants/gene_knockout.py


The 5 successful variant/seed pairs avoid this because they never reach a timestep where `mRNA_unique_index` and the ribosome-derived `unique_mRNA_index_ribosomes` are both empty. In other words, there is still at least one full, translatable mRNA left for the listener to map, so `bulk_name_to_idx(...)` has valid inputs.
1. Dry-run: create variant pickles only (use create_variants.py)
```
python runscripts/create_variants.py --config configs/N_gene_knockout_test.json --kb out/all_media_conditions1/parca/kb -o out/test_variants
```

2. Check out/test_variants/metadata.json and open one .cPickle to confirm sim_data.genetic_perturbations or adjusted arrays were set.

3. Run full workflow (will launch Nextflow)
(If Nextflow/containers not configured, run on a machine with Nextflow installed. Use --resume to restart.)
```
python runscripts/workflow.py --config configs/N_gene_knockout_test.json
```

4. Analyze with gene_screen.py / gene_expression_trace.py after sims finish:
(or run gene_expression_trace.py for a specific gene)
```
python reading/gene_screen.py --project gene_knockout_test --variants 1 2 --generations 1 --gene-list /user/home/il22158/work/vEcoli/reading/results/knockout_experiment/knocked_out_test.txt
```

**Second run test plan**
1. Same procedure as the first run test step 3 and 4, with the problematic gene list (p_lilst) from previous simulation.
2. Run gene screen with the p_list.
3. Without knowing how many of variants run with success status:
```bash
python reading/gene_screen.py --project gene_knockout_p_list --lineage-seed 100 101 --variants $(seq 1 52) --gene-list /user/home/il22158/work/vEcoli/surrogate/results/failure/outlier_genes_all_unique.txt
```
4. After knowing how many of variants run with success status:
```bash
python reading/gene_screen.py --project gene_knockout_p_list --lineage-seed 100 101 --variants 0 1 2 47 48  --gene-list /user/home/il22158/work/vEcoli/surrogate/results/failure/outlier_genes_all_unique.txt
```

## Variant creation 2 knockouts test

In [11]:
import pickle
import numpy as np
import os

base_dir = '/user/home/il22158/work/vEcoli/out/test_variants'

with open(os.path.join(base_dir, '0.cPickle'), 'rb') as f:
    sd_v0 = pickle.load(f)
with open(os.path.join(base_dir, '1.cPickle'), 'rb') as f:
    sd_v1 = pickle.load(f)
with open(os.path.join(base_dir, '2.cPickle'), 'rb') as f:
    sd_v2 = pickle.load(f)

transcription_v0 = sd_v0.process.transcription
transcription_v1 = sd_v1.process.transcription
transcription_v2 = sd_v2.process.transcription

def analyze_variant(v0, v1, variant_name):
    """Compare two transcription objects and return metrics"""
    print("="*100)
    print(f"PARAMETER COMPARISON: {variant_name}")
    print("="*100)

    metrics = {}

    # ============================================================================
    # 1. rna_synth_prob
    # ============================================================================
    print("\n1. rna_synth_prob (dict of arrays)")
    print("-" * 100)

    rna_synth_prob_v0 = v0.rna_synth_prob
    rna_synth_prob_v1 = v1.rna_synth_prob

    keys_v0 = set(rna_synth_prob_v0.keys())
    keys_v1 = set(rna_synth_prob_v1.keys())
    keys_identical = keys_v0 == keys_v1
    print(f"Keys identical?: {keys_identical}")

    max_diff_across_all = 0
    for key in rna_synth_prob_v0.keys():
        arr_v0 = rna_synth_prob_v0[key]
        arr_v1 = rna_synth_prob_v1[key]
        diff = np.abs(arr_v0 - arr_v1)
        max_diff_across_all = max(max_diff_across_all, np.max(diff))

    print(f"Max value difference across all arrays: {max_diff_across_all:.2e}")

    basal_sum_v0 = np.sum(rna_synth_prob_v0['basal'])
    basal_sum_v1 = np.sum(rna_synth_prob_v1['basal'])
    print(f"Sum of 'basal' array: V0={basal_sum_v0:.6f}, V1={basal_sum_v1:.6f}")
    
    metrics['rna_synth_prob'] = {'max_diff': max_diff_across_all, 'sum': basal_sum_v1}

    # ============================================================================
    # 2. rna_expression
    # ============================================================================
    print("\n2. rna_expression (dict of arrays)")
    print("-" * 100)

    rna_expression_v0 = v0.rna_expression
    rna_expression_v1 = v1.rna_expression

    keys_v0 = set(rna_expression_v0.keys())
    keys_v1 = set(rna_expression_v1.keys())
    keys_identical = keys_v0 == keys_v1
    print(f"Keys identical?: {keys_identical}")

    max_diff_across_all = 0
    for key in rna_expression_v0.keys():
        arr_v0 = rna_expression_v0[key]
        arr_v1 = rna_expression_v1[key]
        diff = np.abs(arr_v0 - arr_v1)
        max_diff_across_all = max(max_diff_across_all, np.max(diff))

    print(f"Max value difference across all arrays: {max_diff_across_all:.2e}")

    basal_sum_v0 = np.sum(rna_expression_v0['basal'])
    basal_sum_v1 = np.sum(rna_expression_v1['basal'])
    print(f"Sum of 'basal' array: V0={basal_sum_v0:.6f}, V1={basal_sum_v1:.6f}")
    
    metrics['rna_expression'] = {'max_diff': max_diff_across_all, 'sum': basal_sum_v1}

    # ============================================================================
    # 3. exp_free
    # ============================================================================
    print("\n3. exp_free (1D array)")
    print("-" * 100)

    exp_free_v0 = v0.exp_free
    exp_free_v1 = v1.exp_free

    shape_v0 = exp_free_v0.shape
    shape_v1 = exp_free_v1.shape
    print(f"Shape identical?: {shape_v0 == shape_v1}")

    diff = np.abs(exp_free_v0 - exp_free_v1)
    max_diff = np.max(diff)
    print(f"Max value difference: {max_diff:.2e}")

    sum_v0 = np.sum(exp_free_v0)
    sum_v1 = np.sum(exp_free_v1)
    print(f"Sum of array: V0={sum_v0:.10f}, V1={sum_v1:.10f}")
    
    metrics['exp_free'] = {'max_diff': max_diff, 'sum': sum_v1}

    # ============================================================================
    # 4. exp_ppgpp
    # ============================================================================
    print("\n4. exp_ppgpp (1D array)")
    print("-" * 100)

    exp_ppgpp_v0 = v0.exp_ppgpp
    exp_ppgpp_v1 = v1.exp_ppgpp

    shape_v0 = exp_ppgpp_v0.shape
    shape_v1 = exp_ppgpp_v1.shape
    print(f"Shape identical?: {shape_v0 == shape_v1}")

    diff = np.abs(exp_ppgpp_v0 - exp_ppgpp_v1)
    max_diff = np.max(diff)
    print(f"Max value difference: {max_diff:.2e}")

    sum_v0 = np.sum(exp_ppgpp_v0)
    sum_v1 = np.sum(exp_ppgpp_v1)
    print(f"Sum of array: V0={sum_v0:.10f}, V1={sum_v1:.10f}")
    
    metrics['exp_ppgpp'] = {'max_diff': max_diff, 'sum': sum_v1}
    
    return metrics

# ============================================================================
# Run analysis for Variant 1 (EG10109 KO)
# ============================================================================
metrics_v1 = analyze_variant(transcription_v0, transcription_v1, "Variant 1 (EG10109 KO) vs Baseline")

# ============================================================================
# Run analysis for Variant 2 (EG10196 KO)
# ============================================================================
print("\n\n")
metrics_v2 = analyze_variant(transcription_v0, transcription_v2, "Variant 2 (EG10196 KO) vs Baseline")

# ============================================================================
# Summary Table - Both Variants Side-by-Side
# ============================================================================
print("\n\n" + "="*130)
print("FINAL TABLE: Parameter Changes After Gene Knockout (Both Variants)")
print("="*130)
print(f"{'Parameter':<20} {'Variant 1 (EG10109 KO)':<55} {'Variant 2 (EG10196 KO)':<55}")
print(f"{'':20} {'Structure':<15} {'Values':<15} {'Sum':<15} {'Structure':<15} {'Values':<15} {'Sum':<15}")
print("-" * 130)

params = ['rna_synth_prob', 'rna_expression', 'exp_free', 'exp_ppgpp']
structures = {
    'rna_synth_prob': '✓ Keys identical',
    'rna_expression': '✓ Keys identical',
    'exp_free': '✓ Shape (3277,)',
    'exp_ppgpp': '✓ Shape (3277,)'
}

for param in params:
    v1_max_diff = metrics_v1[param]['max_diff']
    v1_sum = metrics_v1[param]['sum']
    v2_max_diff = metrics_v2[param]['max_diff']
    v2_sum = metrics_v2[param]['sum']
    
    struct = structures[param]
    v1_values = f"✗ Δ {v1_max_diff:.2e}"
    v2_values = f"✗ Δ {v2_max_diff:.2e}"
    v1_sum_str = f"✓ {v1_sum:.6f}"
    v2_sum_str = f"✓ {v2_sum:.6f}"
    
    print(f"{param:<20} {struct:<15} {v1_values:<15} {v1_sum_str:<15} {struct:<15} {v2_values:<15} {v2_sum_str:<15}")

print("="*130)


PARAMETER COMPARISON: Variant 1 (EG10109 KO) vs Baseline

1. rna_synth_prob (dict of arrays)
----------------------------------------------------------------------------------------------------
Keys identical?: True
Max value difference across all arrays: 2.73e-05
Sum of 'basal' array: V0=1.000000, V1=1.000000

2. rna_expression (dict of arrays)
----------------------------------------------------------------------------------------------------
Keys identical?: True
Max value difference across all arrays: 3.85e-06
Sum of 'basal' array: V0=1.000000, V1=1.000000

3. exp_free (1D array)
----------------------------------------------------------------------------------------------------
Shape identical?: True
Max value difference: 1.51e-06
Sum of array: V0=1.0000000000, V1=1.0000000000

4. exp_ppgpp (1D array)
----------------------------------------------------------------------------------------------------
Shape identical?: True
Max value difference: 1.89e-06
Sum of array: V0=1.00000000

- Conclusion:
    The modification of gene expression and systematic renormalization work at the variant creation process.

## 8 Generations 2 Knockouts test

- Inputs:
    Baseline - no knockouts;
    Variant 1 - knockout gene EG10109 (Gene 1);
    Variant 2 - knockout gene EG10196 (Gene 2).

- Results:
    Gene screen for two knockout genes saved in:
        /user/home/il22158/work/vEcoli/reading/results/gene_activity_screen/gene_knockout_test2_v_0
        /user/home/il22158/work/vEcoli/reading/results/gene_activity_screen/gene_knockout_test2_v_1
        /user/home/il22158/work/vEcoli/reading/results/gene_activity_screen/gene_knockout_test2_v_2

    Where we can see the results matches with expectation - as the two genes both expressed (translated) in baseline, while only Gene 2 expressed in Varaint 1 and only Gene 1 expressed in Variant 2.

    Meanwhile, the cell fate also matched with the early simulation (cutomised knockout function) - Variant 1 succeed to generation 8 and Variant 2 doesn't.

    The main difference between tested knockouts and early simulation is that the Variant 2 cell dies sooner - it only succeeds to 3rd generation while the early simulation succeeds to 6th generation.

## Read outputs in p_list KO sim

**Plan for p_list KO Simulation Analysis**

1. **Run multi-seed gene screening** (seeds 100 and 101 across 52 variants)
   - Command: `python reading/gene_screen.py --project gene_knockout_p_list --lineage-seed 100 101 --variants $(seq 1 52) --gene-list outlier_genes_all_unique.txt`
   - Expected output structure: `/out/gene_knockout_p_list/{project}/{variant}/{lineage_seed}/`
   - Outputs: CSV files with gene activity, transcription/translation traces, and plots

2. **Aggregate results** - Run aggregation script after screening completes
   - Command: `python reading/aggregate_knockout_results.py --project gene_knockout_p_list`
   - Optional: add `--output-dir`, `--output-file`, or `--base-path` to override defaults
   - Purpose: Consolidate all 52 variants × 2 seeds results into a single summary CSV
   - Input: Success files from `/out/gene_knockout_p_list/success/experiment_id=gene_knockout_p_list/...`
   - Output (default): `/user/home/il22158/work/vEcoli/surrogate/results/gene_knockout_p_list_success_summary.csv`
   - Format: Similar to existing `ko_runs_with_success_generation.csv` with columns:
     * project, variant, lineage_seed, max_success_generation, duration_sec, final_dry_mass, growth_rate, etc.

3. **Validation & analysis**
   - Verify no figure overlap between seeds
   - Check all 52 variants processed with both seeds (expected: 104 result rows)
   - Confirm CSV format matches reference standard
   - Compare seed 100 vs seed 101 results statistically


## Debug

**Investigation Summary**

The knockout definition itself looks reasonable. In [ecoli/variants/gene_knockout.py](/user/home/il22158/work/vEcoli/ecoli/variants/gene_knockout.py), `apply_variant()` maps each `genes_to_knockout` entry to cistron/RNA indices and calls `sim_data.adjust_final_expression(...)` only when it finds a match. The attached config [configs/N_gene_knockout_p_list.json](/user/work/il22158/vEcoli/configs/N_gene_knockout_p_list.json) is also consistent with that behavior: it applies the `gene_knockout` variant to a 52-gene list and runs two seeds (`100` and `101`).

Error Message Location: [nextflow command log](/user/home/il22158/work/vEcoli/out/gene_knockout_p_list/nextflow/nextflow_workdirs/02/db2dbbe190b7231220777600953584/.command.log)

The failure is downstream in the simulation listener, not in the variant generator. A failed task in the Nextflow work directory showed this traceback from [ecoli/processes/listeners/ribosome_data.py](/user/home/il22158/work/vEcoli/ecoli/processes/listeners/ribosome_data.py):

`IndexError: cannot do a non-empty take from an empty axes.`

That happens at the `bulk_name_to_idx(...)` call where `reduced_to_normal_mRNA_indices` is built. For the failing variants/seeds, the mRNA index arrays are empty enough that `np.take(...)` cannot proceed.

The aggregation output [surrogate/results/gene_knockout_p_list_success_summary.csv](/user/home/il22158/work/vEcoli/surrogate/results/gene_knockout_p_list_success_summary.csv) confirms this is not a summary bug: it contains only 10 successful rows, meaning 5 variant/seed combinations completed successfully while the rest failed earlier in the workflow.

The successful variants do not hit this because they still leave at least one full, translatable mRNA in the listener state when `ribosome_data.py` runs. In that case `mRNA_unique_index` and `unique_mRNA_index_ribosomes` are non-empty, so `bulk_name_to_idx(...)` has valid inputs and never reaches the empty-axis path.

Potential fix: Don't change [ecoli/variants/gene_knockout.py](/user/home/il22158/work/vEcoli/ecoli/variants/gene_knockout.py) as the primary fix. The safer change is to harden [ecoli/processes/listeners/ribosome_data.py](/user/home/il22158/work/vEcoli/ecoli/processes/listeners/ribosome_data.py) against empty mRNA/ribosome index arrays and return zero-filled listener values instead of raising. A small defensive check there should prevent the crash for knockout cases that leave no valid mRNA targets.



## Comparison Run: sucessful run gene list

**Third Round Test: gene knockout list for variants that reached generation 8**

**To-do plan**

1. Prepare the third-round knockout input from the successful generation-8 gene set.
2. Run the simulation with renewed configuration.
3. Run the knockout screening with the same project structure and the new gene list file.
   - Command: `python reading/gene_screen.py --project gene_knockout_3_round_test --lineage-seed 100 101 --variants $(seq 0 50) --gene-list surrogate/third_round_tested_gene_list.txt`
4. Aggregate the results and confirm which variants reach generation 8 again.
5. Compare this round against the previous run to see whether the success pattern is reproducible.

**The tested gene list**

They are extracted as the first 50 gene knockouts which succeed to 8th generation in previous simulation (with customised KO function).

Link: [third_round_tested_gene_list.txt](surrogate/results/third_round_tested_gene_list.txt).

## Simluate the rest of 462 genes

**Gene_knockout function: Cistron lookup and TU conversion**

- The knockout code reads cistron metadata from `sim_data.process.transcription.cistron_data.struct_array`.
- For each gene ID, it matches `cistron["gene_id"]` to find the cistron record.
- It then converts the cistron to the corresponding TU/RNA index or indexes with `transcription.cistron_id_to_rna_indexes(cistron_id)`. (Original doc: [rnas.tsv](/user/home/il22158/work/vEcoli/reconstruction/ecoli/flat/rnas.tsv), coverted table: [operon_classification.tsv](/user/home/il22158/work/vEcoli/reading/results/knockout_experiment/operon_classification.tsv))
- The resolved TU indexes are deduplicated and passed to `sim_data.adjust_final_expression(..., [0.0, ...])` to apply the knockout.
- (**New function added after third test**) If the input already matches an RNA/TU ID, the updated `apply_variant()` can use `rna_data` directly instead of doing the cistron lookup.
- In short: gene ID input is converted to TU indexes through cistron data; TU ID input bypasses that lookup and goes straight to the RNA table.


In [7]:
import pandas as pd
from pathlib import Path

# Working directory
base = "/user/home/il22158/work/vEcoli/"

# Load previous simulation genes from CSV
csv_path = Path(f'{base}surrogate/results/ko_runs_with_success_generation.csv')
df = pd.read_csv(csv_path)

# Parse gene IDs from label column (format: "KO: EG10109")
previous_genes = set()
for label in df['label'].unique():
    if pd.notna(label) and label.startswith('KO: '):
        gene_id = label.replace('KO: ', '').strip()
        previous_genes.add(gene_id)

print(f"Previous simulation genes: {len(previous_genes)}")

# Load exclusion lists
third_round_path = Path(f'{base}surrogate/third_round_tested_gene_list.txt')
with open(third_round_path) as f:
    third_round_genes = set(line.strip() for line in f if line.strip())

outlier_path = Path(f'{base}surrogate/results/failure/outlier_genes_all_unique.txt')
with open(outlier_path) as f:
    outlier_genes = set(line.strip() for line in f if line.strip())

exclusion_genes = third_round_genes | outlier_genes
print(f"3rd round genes: {len(third_round_genes)}")
print(f"Outlier/p_list genes: {len(outlier_genes)}")
print(f"Total exclusions: {len(exclusion_genes)}")

# Calculate new genes (not yet tested)
new_genes = previous_genes - exclusion_genes
print(f"New genes to test (excluding duplicates): {len(new_genes)}")

# Save new gene list to csv file
new_genes_df = pd.DataFrame({'gene_id': list(new_genes)})
new_genes_csv_path = Path(f'{base}surrogate/rest_genes_to_test.csv')
new_genes_df.to_csv(new_genes_csv_path, index=False)
print(f"New gene list saved to: {new_genes_csv_path}")

Previous simulation genes: 462
3rd round genes: 50
Outlier/p_list genes: 52
Total exclusions: 102
New genes to test (excluding duplicates): 360
New gene list saved to: /user/home/il22158/work/vEcoli/surrogate/rest_genes_to_test.csv


In [6]:
# Check genes that failed before gen8 and are NOT in p_list
failed_before_gen8 = df[df['failed_before_gen8'] == 1]['gene_id'].unique()
print(f"Total genes that failed before gen8: {len(failed_before_gen8)}")

# Genes that failed before gen8 but are NOT in the outlier/p_list set
failed_not_in_plist = set(failed_before_gen8) - outlier_genes
print(f"Failed before gen8 but NOT in p_list: {len(failed_not_in_plist)}")
print(f"Genes: {sorted(failed_not_in_plist)}")

# Genes that failed before gen8 AND ARE in p_list
failed_in_plist = set(failed_before_gen8) & outlier_genes
print(f"\nFailed before gen8 AND in p_list: {len(failed_in_plist)}")
print(f"Genes: {sorted(failed_in_plist)}")

# Saved the failed but not in p_list genes to a separate CSV for review
failed_not_in_plist_df = pd.DataFrame({'gene_id': sorted(failed_not_in_plist)})
failed_not_in_plist_csv_path = Path(f'{base}surrogate/failed_before_gen8_not_in_plist.csv')
failed_not_in_plist_df.to_csv(failed_not_in_plist_csv_path, index=False)
print(f"Failed before gen8 but not in p_list genes saved to: {failed_not_in_plist_csv_path}")

Total genes that failed before gen8: 117
Failed before gen8 but NOT in p_list: 65
Genes: ['EG10034', 'EG10063', 'EG10071', 'EG10074', 'EG10078', 'EG10081', 'EG10094', 'EG10097', 'EG10187', 'EG10205', 'EG10208', 'EG10383', 'EG10390', 'EG10407', 'EG10409', 'EG10410', 'EG10444', 'EG10445', 'EG10446', 'EG10447', 'EG10448', 'EG10449', 'EG10450', 'EG10451', 'EG10453', 'EG10492', 'EG10496', 'EG10497', 'EG10532', 'EG10581', 'EG10586', 'EG10709', 'EG10710', 'EG10769', 'EG10770', 'EG10793', 'EG10794', 'EG10796', 'EG10797', 'EG10807', 'EG10810', 'EG10871', 'EG10873', 'EG10878', 'EG10893', 'EG10894', 'EG10895', 'EG10903', 'EG10910', 'EG10912', 'EG10947', 'EG10999', 'EG11000', 'EG11001', 'EG11027', 'EG11028', 'EG11030', 'EG11039', 'EG11043', 'EG11067', 'EG11226', 'EG11575', 'EG11576', 'EG11577', 'G6879']

Failed before gen8 AND in p_list: 52
Genes: ['EG10196', 'EG10707', 'EG10864', 'EG10865', 'EG10866', 'EG10867', 'EG10868', 'EG10869', 'EG10870', 'EG10872', 'EG10874', 'EG10875', 'EG10876', 'EG10877

In [9]:
"""Covert gene ids to TU ids to avoid duplicating tests of genes in the same TU in the rest gene list"""
# Base path
base = "/user/home/il22158/work/vEcoli/"

# Load operon classification and target gene list
operon_path = Path(f'{base}reading/results/knockout_experiment/operon_classification.tsv')
operon_df = pd.read_csv(operon_path, sep='\t')
target_genes_path = Path(f'{base}surrogate/rest_genes_to_test.csv')
target_genes_df = pd.read_csv(target_genes_path)

# Create mapping from gene ID to TU ID
gene_to_tu = dict(zip(operon_df['id'], operon_df['tu_id']))

# Map new genes to TU IDs and deduplicate
tu_ids = set()
genes_found = []
genes_not_found = []

for gene_id in sorted(new_genes):
    if gene_id in gene_to_tu:
        tu_id = gene_to_tu[gene_id]
        tu_ids.add(tu_id)
        genes_found.append(gene_id)
    else:
        genes_not_found.append(gene_id)

print(f"\nGenes found in operon classification: {len(genes_found)}")
print(f"Genes not found: {len(genes_not_found)}")
if genes_not_found:
    print(f"  Not found: {genes_not_found[:5]}..." if len(genes_not_found) > 5 else f"  Not found: {genes_not_found}")

print(f"\nUnique TU IDs to test (operon-level): {len(tu_ids)}")
print(f"Unique TU IDs (sample): {sorted(tu_ids)[:5]}")

# Display the TU list
print(f"\nAll TU IDs for new experiments:")
for tu_id in sorted(tu_ids):
    print(tu_id)
    
# Save TU list to csv file
tu_ids_df = pd.DataFrame({'tu_id': sorted(tu_ids)})
tu_ids_csv_path = Path(f'{base}surrogate/rest_tu_ids_to_test.csv')
tu_ids_df.to_csv(tu_ids_csv_path, index=False)
print(f"TU ID list saved to: {tu_ids_csv_path}")


Genes found in operon classification: 360
Genes not found: 0

Unique TU IDs to test (operon-level): 292
Unique TU IDs (sample): ['EG10016_RNA[c]', 'EG10214_RNA[c]', 'EG10383_RNA[c]', 'EG10982_RNA[c]', 'EG12662_RNA[c]']

All TU IDs for new experiments:
EG10016_RNA[c]
EG10214_RNA[c]
EG10383_RNA[c]
EG10982_RNA[c]
EG12662_RNA[c]
EG12663_RNA[c]
TU0-1002[c]
TU0-1021[c]
TU0-1063[c]
TU0-12797[c]
TU0-12810[c]
TU0-12812[c]
TU0-12827[c]
TU0-12831[c]
TU0-12833[c]
TU0-12921[c]
TU0-12924[c]
TU0-12942[c]
TU0-12947[c]
TU0-12962[c]
TU0-12963[c]
TU0-13010[c]
TU0-13018[c]
TU0-13066[c]
TU0-13069[c]
TU0-13078[c]
TU0-13080[c]
TU0-13084[c]
TU0-13087[c]
TU0-13093[c]
TU0-13100[c]
TU0-13106[c]
TU0-13135[c]
TU0-13182[c]
TU0-13189[c]
TU0-13209[c]
TU0-13215[c]
TU0-13350[c]
TU0-13373[c]
TU0-13440[c]
TU0-13444[c]
TU0-13499[c]
TU0-13514[c]
TU0-13519[c]
TU0-13539[c]
TU0-13552[c]
TU0-13565[c]
TU0-13580[c]
TU0-13593[c]
TU0-13627[c]
TU0-13637[c]
TU0-13642[c]
TU0-13651[c]
TU0-13713[c]
TU0-13736[c]
TU0-13741[c]
TU0-13783[

In [10]:
# Covert the mapping table from tsv to csv
original_file = "/user/home/il22158/work/vEcoli/reading/results/knockout_experiment/operon_classification.tsv"
operon_df = pd.read_csv(original_file, sep='\t')
csv_output_path = "/user/home/il22158/work/vEcoli/surrogate/operon_classification.csv"
operon_df.to_csv(csv_output_path, index=False)
print(f"Operon classification table saved to: {csv_output_path}")

Operon classification table saved to: /user/home/il22158/work/vEcoli/surrogate/operon_classification.csv


In [11]:
from ast import literal_eval

# Check whether the remaining gene list covers all genes in each multi-gene operon.
# This helps verify that moving from gene-level to operon-level does not leave out
# any genes that share the same TU.

def parse_gene_list(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        try:
            parsed = literal_eval(text)
            if isinstance(parsed, list):
                return [item.strip() for item in parsed if isinstance(item, str) and item.strip()]
        except Exception:
            pass
        return [item.strip() for item in text.split(',') if item.strip()]
    return []

# Build the remaining gene set from the notebook table, if needed.
if 'target_genes_df' in globals() and not target_genes_df.empty:
    if 'gene_id' in target_genes_df.columns:
        rest_gene_to_test = set(target_genes_df['gene_id'].astype(str).str.strip())
    elif 'id' in target_genes_df.columns:
        rest_gene_to_test = set(target_genes_df['id'].astype(str).str.strip())
    else:
        rest_gene_to_test = set(target_genes_df.iloc[:, 0].astype(str).str.strip())
else:
    rest_gene_to_test = set(new_genes)

print(f"Remaining genes to test: {len(rest_gene_to_test)}")

# Focus on operons represented in the remaining gene list.
rest_operons = operon_df[operon_df['id'].isin(rest_gene_to_test)].copy()
rest_operons['operon_genes'] = rest_operons.apply(
    lambda row: sorted(set([row['id']] + parse_gene_list(row['other_genes_in_operon']))),
    axis=1,
)

multi_gene_operons = rest_operons[rest_operons['operon_size'].fillna(1).astype(float) > 1].copy()
print(f"Genes in multi-gene operons: {len(multi_gene_operons)}")
print(f"Unique multi-gene operons touched by rest_gene_to_test: {multi_gene_operons['tu_id'].nunique()}")

# Check whether the list covers every gene in each operon.
operon_gaps = []
for tu_id, group in multi_gene_operons.groupby('tu_id'):
    operon_genes = set()
    for _, row in group.iterrows():
        operon_genes.update(row['operon_genes'])
    present_genes = operon_genes & rest_gene_to_test
    missing_genes = sorted(operon_genes - rest_gene_to_test)
    if missing_genes:
        operon_gaps.append({
            'tu_id': tu_id,
            'operon_genes': sorted(operon_genes),
            'present_genes': sorted(present_genes),
            'missing_genes': missing_genes,
        })

print(f"Operons where rest_gene_to_test does NOT cover all genes: {len(operon_gaps)}")

if operon_gaps:
    print("\nExamples of incomplete operon coverage:")
    for gap in operon_gaps[:10]:
        print(f"- {gap['tu_id']}: present={gap['present_genes']}, missing={gap['missing_genes']}")
else:
    print("\nAll multi-gene operons touched by rest_gene_to_test are fully covered.")

# Optional: summarize the operons that are fully covered.
covered_operons = []
for tu_id, group in multi_gene_operons.groupby('tu_id'):
    operon_genes = set()
    for _, row in group.iterrows():
        operon_genes.update(row['operon_genes'])
    if operon_genes.issubset(rest_gene_to_test):
        covered_operons.append((tu_id, sorted(operon_genes)))

print(f"\nFully covered multi-gene operons: {len(covered_operons)}")
for tu_id, genes in covered_operons[:10]:
    print(f"- {tu_id}: {genes}")


Remaining genes to test: 360
Genes in multi-gene operons: 225
Unique multi-gene operons touched by rest_gene_to_test: 157
Operons where rest_gene_to_test does NOT cover all genes: 139

Examples of incomplete operon coverage:
- TU0-1002[c]: present=['G6941', 'G6943'], missing=['G6940', 'G6942', 'G6944']
- TU0-12810[c]: present=['EG12312'], missing=['EG12313', 'EG12314']
- TU0-12827[c]: present=['EG10207'], missing=['EG10570', 'EG11411']
- TU0-12921[c]: present=['G6234'], missing=['EG10666', 'EG10704', 'EG11320', 'EG11321', 'EG11322']
- TU0-12924[c]: present=['G6239'], missing=['G198']
- TU0-12962[c]: present=['EG12384'], missing=['G6287', 'G6288', 'G6289']
- TU0-12963[c]: present=['EG12666'], missing=['EG10758']
- TU0-13010[c]: present=['EG10532'], missing=['EG10855', 'EG11412', 'G6349', 'G6350']
- TU0-13069[c]: present=['G6442'], missing=['G6443']
- TU0-13087[c]: present=['EG10613'], missing=['EG11409', 'EG12375', 'G6471']

Fully covered multi-gene operons: 18
- TU0-12831[c]: ['EG10139

In [14]:
# Build a TU-level output table that keeps track of which remaining genes map to each TU.
# Also flag whether the operon contains genes that are not present in rest_gene_to_test.

tu_to_rest_genes = {}
tu_to_operon_genes = {}
for _, row in operon_df.iterrows():
    gene_id = row['id']
    tu_id = row['tu_id']
    operon_genes = sorted(set([gene_id] + parse_gene_list(row['other_genes_in_operon'])))
    tu_to_operon_genes[tu_id] = operon_genes
    if gene_id in rest_gene_to_test:
        tu_to_rest_genes.setdefault(tu_id, []).append(gene_id)

# Prefer the TU IDs we already derived earlier, but fall back to the keys from the mapping.
final_tu_ids = sorted(tu_ids) if 'tu_ids' in globals() and tu_ids else sorted(tu_to_rest_genes)

# Create the CSV with extra columns showing the genes in rest_gene_to_test that map to each TU
# and whether the operon includes additional genes outside the remaining gene list.
tu_ids_df = pd.DataFrame(
    {
        'tu_id': final_tu_ids,
        'genes_in_rest_gene_to_test': [', '.join(tu_to_rest_genes.get(tu_id, [])) for tu_id in final_tu_ids],
        'n_genes_in_rest_gene_to_test': [len(tu_to_rest_genes.get(tu_id, [])) for tu_id in final_tu_ids],
        'operon_has_additional_genes_not_in_rest_gene_to_test': [
            len(set(tu_to_operon_genes.get(tu_id, [])) - rest_gene_to_test) > 0
            for tu_id in final_tu_ids
        ],
        'n_additional_genes_not_in_rest_gene_to_test': [
            len(set(tu_to_operon_genes.get(tu_id, [])) - rest_gene_to_test)
            for tu_id in final_tu_ids
        ],
        'genes_not_in_rest_gene_to_test': [
            ', '.join(sorted(set(tu_to_operon_genes.get(tu_id, [])) - rest_gene_to_test))
            for tu_id in final_tu_ids
        ],
    }
)

tu_ids_csv_path = Path(f'{base}surrogate/rest_tu_ids_to_test.csv')
tu_ids_df.to_csv(tu_ids_csv_path, index=False)

print(f"TU ID table saved to: {tu_ids_csv_path}")
print(f"Rows written: {len(tu_ids_df)}")
print(tu_ids_df.head(10).to_string(index=False))


TU ID table saved to: /user/home/il22158/work/vEcoli/surrogate/rest_tu_ids_to_test.csv
Rows written: 292
         tu_id genes_in_rest_gene_to_test  n_genes_in_rest_gene_to_test  operon_has_additional_genes_not_in_rest_gene_to_test  n_additional_genes_not_in_rest_gene_to_test genes_not_in_rest_gene_to_test
EG10016_RNA[c]                    EG10016                             1                                                 False                                            0                               
EG10214_RNA[c]                    EG10214                             1                                                 False                                            0                               
EG10383_RNA[c]                    EG10383                             1                                                 False                                            0                               
EG10982_RNA[c]                    EG10982                             1                